1: Instalación de Librerías

---

Este bloque instala todas las librerías necesarias para procesar PDFs, manejar vectores en FAISS, conectar LangSmith y utilizar la API oficial de Google.

In [1]:
!pip install -q langchain langchain-classic langchain-openai langchain-google-genai langchain-community faiss-cpu langsmith python-dotenv PyPDF2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 19.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

2: Credenciales y Configuración

---
carga de forma segura las llaves secretas de Google y activar la trazabilidad obligatoria con LangSmith para el monitoreo del pipeline.


In [2]:
import os
from google.colab import userdata

# Cargar tu clave de Google
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Configurar LangSmith (Trazabilidad)
try:
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "Evaluacion_RAG_Duoc"
    print("✅ Credenciales de Google y LangSmith activadas correctamente.")
except:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("⚠️ LangSmith desactivado, pero Google IA está lista.")

✅ Credenciales de Google y LangSmith activadas correctamente.


3: Procesamiento del Reglamento
---
Este bloque tomara los pdf del colab, y los cortara en fragmentos estratégicos (usando RecursiveCharacterTextSplitter) y los convertirá en vectores usando Google para guardarlos en FAISS.


In [3]:
import PyPDF2
import os
import glob
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print(" Buscando y leyendo los documentos...")

# Busca todos los Archivos PDFs que hay en el Colab
pdf_files = glob.glob("*.pdf")
texto_completo = ""

# Lee cada PDF encontrado
for pdf_path in pdf_files:
    print(f"Procesando: {pdf_path}")
    pdf_reader = PyPDF2.PdfReader(pdf_path)
    texto_completo += "".join([page.extract_text() for page in pdf_reader.pages]) + "\n"

# Divide todo el texto en fragmentos (Chunking)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = text_splitter.split_text(texto_completo)

print("Generando vectores locales con HuggingFace (all-MiniLM-L6-v2)...")

# Creamos los vectores y la base de datos con HuggingFace (100% libre y sin errores de API)
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_texts(texts=chunks, embedding=embeddings_model)

print(f"Listo Documentos procesados y convertidos en {len(chunks)} fragmentos indexados en FAISS.")

/tmp/ipykernel_2168/3134221597.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


 Buscando y leyendo los documentos...
Procesando: Protocolo-para-estudiantes-matriculados.pdf
Procesando: RES-VRA-47-2025-APRUEBA-ACTUALIZACIÓN-DEL-REGLAMENTO-ACADEMICO-ONLINE-1.pdf
Generando vectores locales con HuggingFace (all-MiniLM-L6-v2)...


/tmp/ipykernel_2168/3134221597.py:27: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Listo Documentos procesados y convertidos en 71 fragmentos indexados en FAISS.


4: El Asistente IA

---
Pipeline de recuperación y generación (RAG), procesa la pregunta del usuario, busca los fragmentos más relevantes y aplica guardraíles estrictos con Google Gemini para evitar respuestas fuera de contexto.


In [17]:
import time
import os
from google import genai
from langsmith.run_helpers import traceable

# Inicializar el cliente de gemini
client_genai = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# 1. Función para buscar en la base de datos
@traceable(name="Recuperacion FAISS")
def recuperar_documentos(query):
    retriever = vector_db.as_retriever(search_kwargs={"k": 2})
    return retriever.invoke(query)

# 2. Función para que la IA arme la respuesta (Prompting)
@traceable(name="Generacion LLM")
def generar_respuesta(query, docs):
    contexto = "\n".join([doc.page_content for doc in docs])

    # PROMPT
    prompt = f"""Eres un Asistente Virtual Oficial de Consejería Académica de Duoc UC. Tu tono es profesional, institucional, cercano y empático.
    Tu objetivo es guiar y responder consultas de estudiantes de primer semestre sobre normativas institucionales.

    Basándote principalmente en el siguiente contexto extraído del Reglamento Académico:
    {contexto}

    Pregunta del estudiante: {query}

    INSTRUCCIONES DE RESPUESTA:
    1. Analiza el contexto entregado. Si encuentras información relacionada (aunque sea parcial o conceptual), explica detalladamente lo que estipula el reglamento para orientar al estudiante de forma clara.
    2. Mantén un tono formal pero comprensivo, redactando explicaciones completas y bien desarrolladas (evita respuestas de una sola línea si se puede profundizar con el contexto).
    3. NO inventes artículos que no existan ni asumas normativas externas.
    4. Si la consulta está completamente fuera del alcance de los documentos, responde amablemente: 'Lo siento, no encuentro información específica sobre esa consulta en el Reglamento Académico vigente, te sugiero consultarlo directamente en coordinación de carreras'.
    5. Finaliza señalando que la información está basada en el Reglamento Académico de Duoc UC."""

    # Llamado al modelo
    response = client_genai.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt,
    )
    return response.text, contexto

# ==========================================
# ZONA DE PRUEBAS (Interaccion con la IA)
# ==========================================

# Escribe tu pregunta aquí
pregunta = "Con sus palabra que opina sobre la modalidad online?"

print(f"Estudiante: {pregunta}\n")
print("Buscando en el reglamento...\n")

inicio = time.time()

# Ejecutar pipeline
docs_encontrados = recuperar_documentos(pregunta)
respuesta_final, contexto_usado = generar_respuesta(pregunta, docs_encontrados)

tiempo = time.time() - inicio

print("Asistente Duoc UC:")
print(respuesta_final)
print("-" * 50)
print(f"Tiempo de respuesta: {tiempo:.2f} segundos")

Estudiante: Con sus palabra que opina sobre la modalidad online?

Buscando en el reglamento...

Asistente Duoc UC:
¡Hola! Te doy una cordial bienvenida a Duoc UC y te felicito por dar este importante paso en tu formación académica. 

Como Asistente Virtual de Consejería Académica, más allá de emitir una opinión personal, me complace explicarte cómo nuestra institución concibe y respalda la **modalidad online** desde el punto de vista normativo y formativo.

En Duoc UC, la modalidad online se aborda con la misma rigurosidad, calidad y compromiso que las modalidades presenciales. Según lo estipulado en nuestro marco regulatorio:

1. **Validez y Marco Normativo (Artículo N°2):** El Reglamento Académico es de carácter obligatorio tanto para los y las estudiantes como para el cuerpo docente de las carreras en modalidad online, así como para las unidades académicas asociadas. Esto te garantiza que tu carrera cuenta con un marco claro que protege tus derechos, exige el cumplimiento de tus deb